# CFAR Comparison — CA vs OS vs GO

Classical CFAR detection on the Range-Doppler maps: mechanism, calibration, and a fair
comparison. All scoring goes through the shared `src/eval` module, so these same rules
will later score the deep-learning detector.

Sections:
1. Setup
2. One frame — RD map, CFAR threshold, detections (the visual)
3. Noise estimate & false-alarm calibration
4. **ROC** — the fair comparison (Pd at matched Pfa)
5. Pd vs cell SNR
6. Runtime vs robustness
7. Per-scenario summary

> Key idea: you cannot rank detectors at a single Pfa (a lower threshold trades Pd for Pfa).
> The honest comparison is the ROC and "Pd at a fixed Pfa".

In [ ]:
import sys, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from cfar import CfarConfig, cfar_ratio_map, extract_detections, all_peaks
from eval import MatchConfig, evaluate_frame, aggregate, roc_curve, pd_vs_snr

DATA = ROOT / "data"
MC = MatchConfig(tol_r=1, tol_v=1)
KINDS = ["CA", "OS", "GO"]
COLORS = {"CA": "tab:blue", "OS": "tab:green", "GO": "tab:orange"}

def load_frame(fd):
    power = np.load(fd / "rd_map_mag.npy") ** 2
    lab = pd.read_csv(fd / "labels.csv")
    tru = lab[["range_bin", "velocity_bin"]].to_numpy()
    snr = lab["cell_snr_db"].to_numpy() if "cell_snr_db" in lab else np.full(len(tru), np.nan)
    return power, tru, snr

def frames_of(ds):
    return sorted((DATA / ds / "frames").iterdir())

print("root:", ROOT)
print("scenarios:", [p.name for p in sorted(DATA.iterdir()) if p.is_dir()])

## 2. One frame — RD map, CFAR threshold, and detections

The three panels: the RD power, the CA-CFAR threshold surface (adapts to the local level), and the detections (green) vs true targets (cyan).

In [ ]:
fd = DATA / "clean_debug" / "frames" / "frame_000000"
power, tru, _ = load_frame(fd)
r_ax = np.load(fd / "range_axis.npy"); v_ax = np.load(fd / "velocity_axis.npy")
extent = [v_ax[0], v_ax[-1], r_ax[0], r_ax[-1]]

cfg = CfarConfig(kind="CA", guard=2, train=8, pfa=1e-4)
ratio, valid, alpha = cfar_ratio_map(power, cfg)
det = extract_detections(ratio, valid, alpha, nms_size=5)

power_db = 10*np.log10(power/power.max() + 1e-12)
# threshold in the same relative-dB scale: threshold = alpha * noise, noise = power/ratio
noise = np.where(ratio>0, power/ratio, np.nan)
thr_db = 10*np.log10(alpha*noise/power.max() + 1e-12)

fig, ax = plt.subplots(1, 3, figsize=(19, 5), constrained_layout=True)
for a in ax:
    a.set_xlabel("Velocity [m/s]"); a.set_ylabel("Range [m]"); a.set_ylim(0, 100)
ax[0].imshow(power_db, aspect="auto", origin="lower", extent=extent, cmap="inferno", vmin=-60, vmax=0)
ax[0].set_title("RD power [dB]")
ax[1].imshow(thr_db, aspect="auto", origin="lower", extent=extent, cmap="viridis", vmin=-60, vmax=0)
ax[1].set_title(f"CA-CFAR threshold (Pfa={cfg.pfa:g})")
ax[2].imshow(power_db, aspect="auto", origin="lower", extent=extent, cmap="inferno", vmin=-60, vmax=0)
ax[2].scatter(v_ax[tru[:,1]], r_ax[tru[:,0]], marker="x", s=140, color="cyan", label="true")
if len(det):
    ax[2].scatter(v_ax[det[:,1].astype(int)], r_ax[det[:,0].astype(int)], s=70,
                  facecolors="none", edgecolors="lime", label="CA detections")
ax[2].legend(loc="upper right"); ax[2].set_title("Detections vs truth")
plt.show()
print(f"{len(det)} detections, alpha={alpha:.2f}")

## 3. Noise estimate & false-alarm calibration

On homogeneous noise all estimates should track the true noise power. CA is unbiased (1.0x); OS sits higher (0.75-percentile of exponential = ln4 = 1.39x); GO slightly high (max of two half-means). The measured Pfa on noise-only frames validates the alpha->Pfa formula.

In [ ]:
power_n, _, _ = load_frame(DATA / "noise_only_debug" / "frames" / "frame_000000")
rows = []
for k in KINDS:
    cfg = CfarConfig(kind=k, guard=2, train=8, pfa=1e-4)
    ratio, valid, alpha = cfar_ratio_map(power_n, cfg)
    est = power_n[valid] / np.where(ratio[valid] > 0, ratio[valid], np.nan)
    rows.append({"method": k, "noise_est/true": np.nanmean(est)/power_n[valid].mean(), "alpha": alpha})
pd.DataFrame(rows).set_index("method")

In [ ]:
# Measured Pfa on ALL noise-only frames, at the design operating point.
rows = []
for k in KINDS:
    cfg = CfarConfig(kind=k, guard=2, train=8, pfa=1e-4)
    evs = []
    for fd in frames_of("noise_only_debug"):
        power, tru, _ = load_frame(fd)
        ratio, valid, alpha = cfar_ratio_map(power, cfg)
        det = extract_detections(ratio, valid, alpha, nms_size=5)
        evs.append(evaluate_frame(det, tru, power.shape, MC, valid_mask=valid))
    a = aggregate(evs)
    rows.append({"method": k, "design_Pfa": 1e-4, "measured_Pfa/cell": a["pfa_per_cell"], "FA/frame": a["fa_per_frame"]})
pd.DataFrame(rows).set_index("method")

## 4. ROC — the fair comparison

For each method we keep ALL candidate peaks with their ratio score, then sweep the score
threshold. Each setting gives one (Pfa, Pd) point. Because the three ratio statistics live on
different scales, we sweep by **target Pfa** and read back the measured Pfa — so all methods are
compared at the **same measured false-alarm rate**, which is the only fair way to rank them.

In [ ]:
# Build candidate peaks once per method on a mixed frame set (multi_target = the interesting case).
def collect(ds, k, cfg):
    out = []
    for fd in frames_of(ds):
        power, tru, snr = load_frame(fd)
        ratio, valid, _ = cfar_ratio_map(power, cfg)
        peaks = all_peaks(ratio, valid, nms_size=5)  # (r, v, score)
        out.append((peaks, tru, power.shape, valid, snr))
    return out

# thresholds: a log range of ratio values spanning the score distributions
thr = np.unique(np.concatenate([np.linspace(1, 60, 40), np.logspace(np.log10(60), np.log10(2000), 20)]))

roc = {}
for k in KINDS:
    cfg = CfarConfig(kind=k, guard=2, train=8)
    data = collect("multi_target_debug", k, cfg)
    frames = [(p, t, s, v) for (p, t, s, v, _) in data]   # (det, truths, shape, valid_mask)
    roc[k] = roc_curve(frames, thresholds=thr, cfg=MC)

fig, ax = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for k in KINDS:
    d = roc[k].sort_values("pfa_per_cell")
    ax[0].plot(d["pfa_per_cell"].clip(1e-6), d["pd"], "-o", ms=3, color=COLORS[k], label=k)
    ax[1].plot(d["fa_per_frame"], d["pd"], "-o", ms=3, color=COLORS[k], label=k)
ax[0].set_xscale("log"); ax[0].set_xlabel("Pfa per cell"); ax[0].set_ylabel("Pd"); ax[0].set_title("ROC (multi_target)")
ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_ylim(0,1)
ax[1].set_xlabel("False alarms per frame"); ax[1].set_ylabel("Pd"); ax[1].set_title("FROC (multi_target)")
ax[1].legend(); ax[1].grid(alpha=.3); ax[1].set_ylim(0,1)
plt.show()

In [ ]:
# Pd at a fixed operating Pfa (~1e-3 per cell): interpolate each ROC.
def pd_at_pfa(df, target=1e-3):
    d = df.sort_values("pfa_per_cell")
    return float(np.interp(target, d["pfa_per_cell"], d["pd"]))
pd.DataFrame([{"method": k, "Pd @ Pfa=1e-3": pd_at_pfa(roc[k])} for k in KINDS]).set_index("method")

## 5. Pd vs cell SNR

Detection probability against each target's own measured cell SNR, at a fixed operating point. The S-curve is the fundamental detector characteristic.

In [ ]:
edges = [0,3,6,9,12,15,18,21,24]
fig, ax = plt.subplots(figsize=(8,5))
for k in KINDS:
    cfg = CfarConfig(kind=k, guard=2, train=8, pfa=1e-3)
    frames = []
    for fd in frames_of("snr_sweep_debug"):
        power, tru, snr = load_frame(fd)
        ratio, valid, alpha = cfar_ratio_map(power, cfg)
        det = extract_detections(ratio, valid, alpha, nms_size=5)
        frames.append((det, tru, power.shape, snr))
    curve = pd_vs_snr(frames, snr_edges=edges, cfg=MC)
    ax.plot(curve["mean_snr_db"], curve["pd"], "-o", color=COLORS[k], label=k)
ax.set_xlabel("cell SNR [dB]"); ax.set_ylabel("Pd"); ax.set_title("Pd vs SNR (snr_sweep)")
ax.grid(alpha=.3); ax.legend(); ax.set_ylim(0,1.02)
plt.show()

## 6. Runtime vs robustness

Mean processing time per frame. OS's robustness (rank filter) costs an order of magnitude more than CA/GO — a real trade-off for the runtime metric.

In [ ]:
rows = []
for k in KINDS:
    cfg = CfarConfig(kind=k, guard=2, train=8, pfa=1e-4)
    t0 = time.perf_counter(); n = 0
    for fd in frames_of("clutter_debug"):
        power, tru, _ = load_frame(fd)
        ratio, valid, alpha = cfar_ratio_map(power, cfg)
        _ = extract_detections(ratio, valid, alpha, nms_size=5); n += 1
    rows.append({"method": k, "ms/frame": (time.perf_counter()-t0)/n*1000})
pd.DataFrame(rows).set_index("method")

## 7. Per-scenario summary

Pd and false-alarms/frame at the design point. Read alongside the ROC — a single-point table can mislead (see the notes above), but it shows the scenario-by-scenario picture.

In [ ]:
rows = []
for ds in ["clean_debug","clutter_debug","multi_target_debug","close_targets_debug","multipath_debug"]:
    row = {"scenario": ds}
    for k in KINDS:
        cfg = CfarConfig(kind=k, guard=2, train=8, pfa=1e-4)
        evs = []
        for fd in frames_of(ds):
            power, tru, _ = load_frame(fd)
            ratio, valid, alpha = cfar_ratio_map(power, cfg)
            det = extract_detections(ratio, valid, alpha, nms_size=5)
            evs.append(evaluate_frame(det, tru, power.shape, MC, valid_mask=valid))
        a = aggregate(evs)
        row[f"{k}_Pd"] = round(a["pd"], 2); row[f"{k}_FA"] = round(a["fa_per_frame"], 1)
    rows.append(row)
pd.DataFrame(rows).set_index("scenario")

## Notes for the report

- **Fair comparison = ROC**, not a single Pfa. At matched Pfa the ranking is meaningful.
- `close_targets` Pd caps at ~0.5 for every method — a resolution limit (weak partner inside
  the mainlobe), not a detector failure.
- CA is unbiased but false-alarms most; OS is robust to interfering targets (multi/close) at a
  runtime cost; GO controls clutter-edge false alarms (Doppler-axis split).
- Guard = 2 (mainlobe cover, measured); training = 8 is a starting point — the CFAR parameter
  sensitivity sweep (Phase D) sets the final values, and calibrates the OS/GO thresholds to an
  exact Pfa.